In [1]:
# ============================================================
# 00 — CONNECT GOOGLE DRIVE / PROJECT
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [13]:
# ============================================================
# PROJECT PATH
# ============================================================

from pathlib import Path
import sys

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FitnessML_Master"
)

print(f"Project directory: {PROJECT_DIR}")
print(f"Exists: {PROJECT_DIR.exists()}")

Project directory: /content/drive/MyDrive/FitnessML_Master
Exists: True


In [14]:
# ============================================================
# ADD PROJECT TO PYTHON PATH
# ============================================================

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print("Project added to Python path.")

Project added to Python path.


In [15]:
# ============================================================
# 005 — MACHINE LEARNING MODELS
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FitnessML_Master"
)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import config_fitbit as config

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("=" * 70)
print("FITBIT MACHINE LEARNING EXPERIMENTS")
print("=" * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FITBIT MACHINE LEARNING EXPERIMENTS


In [16]:
# ============================================================
# LOAD MODELING DATASET
# ============================================================

model_path = (
    Path(config.RAW_DATA_DIR).parents[1]
    / "processed"
    / "fitbit_modeling_features.csv"
)

model_df = pd.read_csv(model_path)

model_df["ActivityDate"] = pd.to_datetime(
    model_df["ActivityDate"]
)

model_df = (
    model_df
    .sort_values(["ActivityDate", "Id"])
    .reset_index(drop=True)
)

print(f"Dataset shape: {model_df.shape}")
print(f"Users: {model_df['Id'].nunique()}")

print(
    f"Date range: "
    f"{model_df['ActivityDate'].min().date()} → "
    f"{model_df['ActivityDate'].max().date()}"
)

display(model_df.head())

Dataset shape: (680, 68)
Users: 32
Date range: 2016-04-19 → 2016-05-11


,Id,ActivityDate,TotalSteps,TotalDistance,TrackerDistance,LoggedActivitiesDistance,VeryActiveDistance,ModeratelyActiveDistance,LightActiveDistance,SedentaryActiveDistance,VeryActiveMinutes,FairlyActiveMinutes,LightlyActiveMinutes,SedentaryMinutes,Calories,Calories_lag_1,Calories_lag_2,Calories_lag_3,Calories_lag_7,TotalSteps_lag_1,TotalSteps_lag_2,TotalSteps_lag_3,TotalSteps_lag_7,TotalDistance_lag_1,TotalDistance_lag_2,TotalDistance_lag_3,TotalDistance_lag_7,VeryActiveMinutes_lag_1,VeryActiveMinutes_lag_2,VeryActiveMinutes_lag_3,VeryActiveMinutes_lag_7,FairlyActiveMinutes_lag_1,FairlyActiveMinutes_lag_2,FairlyActiveMinutes_lag_3,FairlyActiveMinutes_lag_7,LightlyActiveMinutes_lag_1,LightlyActiveMinutes_lag_2,LightlyActiveMinutes_lag_3,LightlyActiveMinutes_lag_7,SedentaryMinutes_lag_1,SedentaryMinutes_lag_2,SedentaryMinutes_lag_3,SedentaryMinutes_lag_7,Calories_rolling_mean_3,Calories_rolling_std_3,Calories_rolling_mean_7,Calories_rolling_std_7,TotalSteps_rolling_mean_3,TotalSteps_rolling_std_3,TotalSteps_rolling_mean_7,TotalSteps_rolling_std_7,TotalDistance_rolling_mean_3,TotalDistance_rolling_std_3,TotalDistance_rolling_mean_7,TotalDistance_rolling_std_7,VeryActiveMinutes_rolling_mean_3,VeryActiveMinutes_rolling_std_3,VeryActiveMinutes_rolling_mean_7,VeryActiveMinutes_rolling_std_7,LightlyActiveMinutes_rolling_mean_3,LightlyActiveMinutes_rolling_std_3,LightlyActiveMinutes_rolling_mean_7,LightlyActiveMinutes_rolling_std_7,SedentaryMinutes_rolling_mean_3,SedentaryMinutes_rolling_std_3,SedentaryMinutes_rolling_mean_7,SedentaryMinutes_rolling_std_7,target_calories_next_day
0,1503960366,2016-04-19,15506,9.88,9.88,0.0,3.53,1.32,5.03,0.0,50,31,264,775,2035,1921.0,1728.0,1863.0,1985.0,13019.0,9705.0,12669.0,13162.0,8.59,6.48,8.16,8.50,42.0,38.0,36.0,25.0,16.0,20.0,10.0,13.0,233.0,164.0,221.0,328.0,1149.0,539.0,773.0,728.0,1837.333333,99.026932,1830.714286,95.764841,11797.666667,1820.732087,11358.857143,1538.733833,7.743333,1.115004,7.388571,0.993654,38.666667,3.055050,31.571429,7.457818,206.000000,36.864617,221.857143,52.594133,820.333333,307.742316,844.142857,245.957604,1786.0
1,1624580081,2016-04-19,2916,1.90,1.90,0.0,0.00,0.00,1.90,0.0,0,0,141,1299,1435,1604.0,1554.0,1463.0,1432.0,10536.0,6175.0,5370.0,8163.0,7.41,4.06,3.49,5.31,17.0,15.0,0.0,0.0,7.0,22.0,0.0,0.0,202.0,127.0,176.0,146.0,1214.0,1276.0,1264.0,1294.0,1540.333333,71.486595,1482.857143,95.837412,7360.333333,2779.505412,6838.285714,2932.289419,4.986667,2.117931,4.531429,2.030611,10.666667,9.291573,4.571429,7.828519,168.333333,38.083242,161.571429,47.088063,1251.333333,32.883633,1269.714286,48.475816,1446.0
2,1644430081,2016-04-19,11256,8.18,8.18,0.0,0.36,2.53,5.30,0.0,5,58,278,1099,3300,2806.0,3011.0,3493.0,3199.0,7132.0,8757.0,15300.0,10694.0,5.19,6.37,11.12,7.77,15.0,29.0,51.0,2.0,33.0,13.0,42.0,51.0,121.0,186.0,212.0,256.0,1271.0,1212.0,1135.0,1131.0,3103.333333,352.684467,3055.285714,265.264468,10396.333333,4323.727134,9454.857143,3260.173381,7.560000,3.138997,6.874286,2.366832,31.666667,18.147543,19.285714,18.263938,173.000000,46.872167,190.285714,53.049393,1206.000000,68.198240,1199.428571,67.384116,2430.0
3,1844505072,2016-04-19,197,0.13,0.13,0.0,0.00,0.00,0.13,0.0,0,0,10,1430,1366,1814.0,1793.0,1657.0,2030.0,4597.0,4525.0,3414.0,6697.0,3.04,2.99,2.26,4.43,0.0,2.0,0.0,0.0,12.0,8.0,0.0,0.0,217.0,199.0,147.0,339.0,1211.0,1231.0,1293.0,1101.0,1754.666667,85.231059,1858.428571,167.243193,4178.666667,663.198563,5134.714286,1613.668152,2.763333,0.436616,3.395714,1.067690,0.666667,1.154701,0.285714,0.755929,187.666667,36.350149,242.714286,84.120379,1245.000000,42.755117,1056.857143,275.845626,1349.0
4,1927972279,2016-04-19,0,0.00,0.00,0.0,0.00,0.00,0.00,0.0,0,0,0,1440,2063,2111.0,2063.0,2064.0,2220.0,244.0,0.0,0.0,678.0,0.17,0.00,0.00,0.47,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,55.0,1423.0,1440.0,1440.0,734.0,2079.333333,27.428695,2173.285714,113.226490,81.333333,140.873466,631.571429,762.970916,0.056667,0.098150,0.438571,0.528880,0.000000,0.000000,0.000000,0.0000

In [17]:
# ============================================================
# FEATURE GROUPS
# ============================================================

target_column = "target_calories_next_day"

lag_features = [
    c for c in model_df.columns
    if "_lag_" in c
]

rolling_features = [
    c for c in model_df.columns
    if "rolling_" in c
]

temporal_features = (
    lag_features +
    rolling_features
)

# Current-day features
base_features = [
    c for c in model_df.columns
    if c not in [
        "Id",
        "ActivityDate",
        target_column
    ]
    and "_lag_" not in c
    and "rolling_" not in c
]

print("=" * 70)
print("FEATURE GROUPS")
print("=" * 70)

print(f"Base features:     {len(base_features)}")
print(f"Lag features:      {len(lag_features)}")
print(f"Rolling features:  {len(rolling_features)}")
print(f"Temporal features: {len(temporal_features)}")

FEATURE GROUPS
Base features:     13
Lag features:      28
Rolling features:  24
Temporal features: 52


In [18]:
# ============================================================
# TEMPORAL TRAIN / TEST SPLIT
# ============================================================

unique_dates = np.sort(
    model_df["ActivityDate"].unique()
)

split_index = int(
    len(unique_dates) * 0.80
)

split_date = unique_dates[split_index]

train_df = model_df[
    model_df["ActivityDate"] < split_date
].copy()

test_df = model_df[
    model_df["ActivityDate"] >= split_date
].copy()

print("=" * 70)
print("TEMPORAL TRAIN / TEST SPLIT")
print("=" * 70)

print(f"Split date: {pd.Timestamp(split_date).date()}")

print()
print(
    f"Train: {len(train_df):,} rows | "
    f"{train_df['ActivityDate'].min().date()} → "
    f"{train_df['ActivityDate'].max().date()}"
)

print(
    f"Test:  {len(test_df):,} rows | "
    f"{test_df['ActivityDate'].min().date()} → "
    f"{test_df['ActivityDate'].max().date()}"
)

TEMPORAL TRAIN / TEST SPLIT
Split date: 2016-05-07

Train: 555 rows | 2016-04-19 → 2016-05-06
Test:  125 rows | 2016-05-07 → 2016-05-11


In [19]:
# ============================================================
# MODEL EVALUATION
# ============================================================

def evaluate_model(
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name
):

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(
        y_test,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )

    r2 = r2_score(
        y_test,
        predictions
    )

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [20]:
# ============================================================
# EXPERIMENT A — BASE FEATURES
# ============================================================

X_train = train_df[base_features]
X_test = test_df[base_features]

y_train = train_df[target_column]
y_test = test_df[target_column]

results = []

results.append(
    evaluate_model(
        LinearRegression(),
        X_train,
        y_train,
        X_test,
        y_test,
        "Linear Regression — Base"
    )
)

results.append(
    evaluate_model(
        DecisionTreeRegressor(
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train,
        X_test,
        y_test,
        "Decision Tree — Base"
    )
)

print("=" * 70)
print("EXPERIMENT A — BASE FEATURES")
print("=" * 70)

display(
    pd.DataFrame(results).round(4)
)

EXPERIMENT A — BASE FEATURES


,Model,MAE,RMSE,R2
0,Linear Regression — Base,496.0381,738.5981,0.2541
1,Decision Tree — Base,496.2134,719.3816,0.2924


In [21]:
# ============================================================
# EXPERIMENT B — LAG FEATURES
# ============================================================

lag_only_features = (
    lag_features
)

X_train = train_df[lag_only_features]
X_test = test_df[lag_only_features]

results.append(
    evaluate_model(
        LinearRegression(),
        X_train,
        y_train,
        X_test,
        y_test,
        "Linear Regression — Lag"
    )
)

results.append(
    evaluate_model(
        DecisionTreeRegressor(
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train,
        X_test,
        y_test,
        "Decision Tree — Lag"
    )
)

display(
    pd.DataFrame(results).round(4)
)

,Model,MAE,RMSE,R2
0,Linear Regression — Base,496.0381,738.5981,0.2541
1,Decision Tree — Base,496.2134,719.3816,0.2924
2,Linear Regression — Lag,503.1245,760.2357,0.2098
3,Decision Tree — Lag,566.1334,784.0076,0.1596


In [22]:
# ============================================================
# EXPERIMENT C — ROLLING FEATURES
# ============================================================

X_train = train_df[rolling_features]
X_test = test_df[rolling_features]

results.append(
    evaluate_model(
        LinearRegression(),
        X_train,
        y_train,
        X_test,
        y_test,
        "Linear Regression — Rolling"
    )
)

results.append(
    evaluate_model(
        DecisionTreeRegressor(
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train,
        X_test,
        y_test,
        "Decision Tree — Rolling"
    )
)

display(
    pd.DataFrame(results).round(4)
)

,Model,MAE,RMSE,R2
0,Linear Regression — Base,496.0381,738.5981,0.2541
1,Decision Tree — Base,496.2134,719.3816,0.2924
2,Linear Regression — Lag,503.1245,760.2357,0.2098
3,Decision Tree — Lag,566.1334,784.0076,0.1596
4,Linear Regression — Rolling,492.4569,758.8200,0.2127
5,Decision Tree — Rolling,517.9900,787.8379,0.1513


In [23]:
# ============================================================
# EXPERIMENT D — BASE + LAG + ROLLING
# ============================================================

all_features = (
    base_features +
    lag_features +
    rolling_features
)

X_train = train_df[all_features]
X_test = test_df[all_features]

results.append(
    evaluate_model(
        LinearRegression(),
        X_train,
        y_train,
        X_test,
        y_test,
        "Linear Regression — All Features"
    )
)

results.append(
    evaluate_model(
        DecisionTreeRegressor(
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train,
        X_test,
        y_test,
        "Decision Tree — All Features"
    )
)

display(
    pd.DataFrame(results).round(4)
)

,Model,MAE,RMSE,R2
0,Linear Regression — Base,496.0381,738.5981,0.2541
1,Decision Tree — Base,496.2134,719.3816,0.2924
2,Linear Regression — Lag,503.1245,760.2357,0.2098
3,Decision Tree — Lag,566.1334,784.0076,0.1596
4,Linear Regression — Rolling,492.4569,758.8200,0.2127
5,Decision Tree — Rolling,517.9900,787.8379,0.1513
6,Linear Regression — All Features,485.0494,754.3263,0.2220
7,Decision Tree — All Features,533.4941,789.1613,0.1485


In [24]:
# ============================================================
# EXPERIMENT E — RANDOM FOREST
# ============================================================

X_train = train_df[all_features]
X_test = test_df[all_features]

results.append(
    evaluate_model(
        RandomForestRegressor(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1
        ),
        X_train,
        y_train,
        X_test,
        y_test,
        "Random Forest — All Features"
    )
)

results_df = pd.DataFrame(results)

print("=" * 70)
print("ML RESULTS")
print("=" * 70)

display(
    results_df
    .sort_values("RMSE")
    .round(4)
)

ML RESULTS


,Model,MAE,RMSE,R2
1,Decision Tree — Base,496.2134,719.3816,0.2924
0,Linear Regression — Base,496.0381,738.5981,0.2541
8,Random Forest — All Features,481.7334,740.4334,0.2504
6,Linear Regression — All Features,485.0494,754.3263,0.2220
4,Linear Regression — Rolling,492.4569,758.8200,0.2127
2,Linear Regression — Lag,503.1245,760.2357,0.2098
3,Decision Tree — Lag,566.1334,784.0076,0.1596
5,Decision Tree — Rolling,517.9900,787.8379,0.1513
7,Decision Tree — All Features,533.4941,789.1613,0.1485


In [25]:
# ============================================================
# TEMPORAL BASELINES
# ============================================================

print("=" * 70)
print("TEMPORAL BASELINES")
print("=" * 70)

baseline_results = []

# ------------------------------------------------------------
# 1. GLOBAL MEAN
# ------------------------------------------------------------

global_mean = train_df["Calories"].mean()

global_predictions = np.full(
    len(test_df),
    global_mean
)

baseline_results.append({
    "Model": "Global Mean",
    "MAE": mean_absolute_error(
        y_test,
        global_predictions
    ),
    "RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            global_predictions
        )
    ),
    "R2": r2_score(
        y_test,
        global_predictions
    )
})


# ------------------------------------------------------------
# 2. USER MEAN
# ------------------------------------------------------------

user_means = (
    train_df
    .groupby("Id")["Calories"]
    .mean()
)

user_predictions = (
    test_df["Id"]
    .map(user_means)
    .fillna(global_mean)
)

baseline_results.append({
    "Model": "User Mean",
    "MAE": mean_absolute_error(
        y_test,
        user_predictions
    ),
    "RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            user_predictions
        )
    ),
    "R2": r2_score(
        y_test,
        user_predictions
    )
})


# ------------------------------------------------------------
# 3. PREVIOUS DAY
# ------------------------------------------------------------

previous_day_predictions = test_df["Calories_lag_1"]

baseline_results.append({
    "Model": "Previous Day",
    "MAE": mean_absolute_error(
        y_test,
        previous_day_predictions
    ),
    "RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            previous_day_predictions
        )
    ),
    "R2": r2_score(
        y_test,
        previous_day_predictions
    )
})


# ------------------------------------------------------------
# 4. 7-DAY MEAN
# ------------------------------------------------------------

seven_day_predictions = (
    test_df["Calories_rolling_mean_7"]
)

baseline_results.append({
    "Model": "7-Day Mean",
    "MAE": mean_absolute_error(
        y_test,
        seven_day_predictions
    ),
    "RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            seven_day_predictions
        )
    ),
    "R2": r2_score(
        y_test,
        seven_day_predictions
    )
})


baseline_results_df = pd.DataFrame(
    baseline_results
)

print("=" * 70)
print("BASELINE RESULTS")
print("=" * 70)

display(
    baseline_results_df
    .sort_values("RMSE")
    .round(4)
)

TEMPORAL BASELINES
BASELINE RESULTS


,Model,MAE,RMSE,R2
1,User Mean,486.7907,753.1806,0.2244
3,7-Day Mean,479.9669,757.9682,0.2145
2,Previous Day,542.4320,803.8786,0.1164
0,Global Mean,732.1124,898.7329,-0.1044


In [26]:
# ============================================================
# COMPLETE EXPERIMENT COMPARISON
# ============================================================

all_results = pd.concat(
    [
        baseline_results_df,
        results_df
    ],
    ignore_index=True
)

print("=" * 70)
print("COMPLETE EXPERIMENT COMPARISON")
print("=" * 70)

display(
    all_results
    .sort_values("RMSE")
    .round(4)
)

COMPLETE EXPERIMENT COMPARISON


,Model,MAE,RMSE,R2
5,Decision Tree — Base,496.2134,719.3816,0.2924
4,Linear Regression — Base,496.0381,738.5981,0.2541
12,Random Forest — All Features,481.7334,740.4334,0.2504
1,User Mean,486.7907,753.1806,0.2244
10,Linear Regression — All Features,485.0494,754.3263,0.2220
3,7-Day Mean,479.9669,757.9682,0.2145
8,Linear Regression — Rolling,492.4569,758.8200,0.2127
6,Linear Regression — Lag,503.1245,760.2357,0.2098
7,Decision Tree — Lag,566.1334,784.0076,0.1596
9,Decision Tree — Rolling,517.9900,787.8379,0.1513


In [27]:
# ============================================================
# SAVE 005 RESULTS
# ============================================================

tables_dir = PROJECT_DIR / "tables"
reports_dir = PROJECT_DIR / "reports"

tables_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Full comparison
results_path = tables_dir / "005_ml_results.csv"

all_results.to_csv(
    results_path,
    index=False
)

# Baselines separately
baseline_path = tables_dir / "005_baseline_results.csv"

baseline_results_df.to_csv(
    baseline_path,
    index=False
)

# Human-readable report
report_path = reports_dir / "005_ml_results.md"

with open(report_path, "w", encoding="utf-8") as f:

    f.write("# 005 — Machine Learning Results\n\n")

    f.write("## Dataset\n\n")
    f.write("- Dataset: Fitbit Fitness Tracker\n")
    f.write("- Target: `target_calories_next_day`\n")
    f.write("- Temporal split: 80/20\n")
    f.write("- Split date: 2016-05-07\n")
    f.write("- Train: 555 rows\n")
    f.write("- Test: 125 rows\n\n")

    f.write("## Complete Experiment Comparison\n\n")
    f.write(
        all_results
        .sort_values("RMSE")
        .round(4)
        .to_markdown(index=False)
    )

    f.write("\n\n## Main observations\n\n")
    f.write(
        "- Best R²: Decision Tree — Base (0.2924)\n"
        "- Best RMSE: Decision Tree — Base (719.38)\n"
        "- Best MAE: 7-Day Mean (479.97)\n"
        "- Random Forest — All Features: MAE 481.73, R² 0.2504\n"
        "- Temporal lag/rolling features did not improve the main models.\n"
        "- User Mean was a strong baseline (R² 0.2244).\n"
    )

print("=" * 70)
print("005 RESULTS SAVED")
print("=" * 70)

print(f"ML results:       {results_path}")
print(f"Baseline results: {baseline_path}")
print(f"Report:           {report_path}")

005 RESULTS SAVED
ML results:       /content/drive/MyDrive/FitnessML_Master/tables/005_ml_results.csv
Baseline results: /content/drive/MyDrive/FitnessML_Master/tables/005_baseline_results.csv
Report:           /content/drive/MyDrive/FitnessML_Master/reports/005_ml_results.md
